# 🔗 네이버 검색 API로 데이터 수집하기 (VSCode 실습용)

---

이 노트북은 **네이버 개발자 센터(NAVER Developers)**에서 제공하는
**검색 API**를 활용하여 블로그, 뉴스, 웹문서 등의 데이터를 수집하는 방법을 학습합니다.

### 학습 로드맵

| 순서 | 주제 | 핵심 내용 |
|------|------|-----------|
| 0 | API란 무엇인가? | API 개념, REST API, 인증 방식 |
| 1 | 네이버 검색 API 등록 | 애플리케이션 등록, Client ID/Secret 발급 |
| 2 | VSCode 환경 설정 | 가상환경, 패키지 설치, 인증 정보 관리 |
| 3 | API 인증 설정 | .env 파일, 환경 변수 |
| 4 | 첫 번째 API 호출 | 웹문서 검색, 요청/응답 구조 이해 |
| 5 | 응답 데이터 분석 | JSON 구조, DataFrame 변환 |
| 6 | 검색 카테고리 | 10가지 검색 카테고리와 endpoint |
| 7 | 통합 검색 클래스 | 페이지네이션, 다중 카테고리 지원 |
| 8 | 실행 및 저장 | 검색 수집 실행, Excel 저장 |
| - | Quick Reference | 핵심 API 요약표 |

## 0. API란 무엇인가?

### API (Application Programming Interface)

**API**는 프로그램끼리 데이터를 주고받기 위한 **약속된 인터페이스**입니다.
네이버 검색 API를 사용하면 브라우저 없이 **Python 코드로 직접 검색 결과**를 받아올 수 있습니다.

```
[내 Python 코드]  →  HTTP 요청 (검색어, 인증키)  →  [네이버 API 서버]
                 ←  HTTP 응답 (JSON 데이터)      ←
```

### 스크래핑 vs API 비교

| 비교 항목 | 웹 스크래핑 | API 사용 |
|-----------|------------|----------|
| 데이터 형식 | HTML (파싱 필요) | **JSON** (구조화된 데이터) |
| 안정성 | HTML 구조 변경 시 깨짐 | **안정적** (공식 인터페이스) |
| 법적 이슈 | 사이트 정책에 따라 제한 | **공식 허용** (약관 내) |
| 인증 | 불필요 (User-Agent만) | **API 키 필요** (등록 필요) |
| 수집량 제한 | 서버 부하 주의 | **일일 25,000건** (네이버 기준) |

### REST API 기본 구조

```python
import requests

response = requests.get(
    url     = "https://openapi.naver.com/v1/search/blog.json",  # API 엔드포인트
    headers = {"X-Naver-Client-Id": "...", ...},                 # 인증 정보
    params  = {"query": "검색어", "display": 10}                 # 요청 파라미터
)

data = response.json()   # JSON 응답 → Python 딕셔너리
```

### 네이버 검색 API 요청 흐름

```
1. 네이버 개발자 센터에서 애플리케이션 등록
2. Client ID / Client Secret 발급
3. HTTP GET 요청 (URL + 헤더 + 파라미터)
4. JSON 응답 수신
5. 데이터 추출 및 DataFrame 변환
```

## 1. 네이버 검색 API 등록 (Client ID/Secret 발급)

### 등록 절차

1. **네이버 로그인** 후 [네이버 개발자 센터](https://developers.naver.com/) 접속

2. **애플리케이션 등록**: [등록 페이지](https://developers.naver.com/docs/common/openapiguide/appregister.md)
   - 애플리케이션 이름: 자유롭게 설정 (예: `my_search_app`)
   - 사용 API: **검색** 선택
   - 비로그인 오픈 API 서비스 환경: **WEB 설정**
   - 웹서비스 URL: `http://localhost`

3. **Client ID / Client Secret 확인**
   - 등록 완료 후 **내 애플리케이션** 페이지에서 확인 가능
   - 이 두 값이 API 인증에 사용됩니다

### API 사용 한도

| 항목 | 제한 |
|------|------|
| 일일 호출 한도 | **25,000건** |
| 1회 최대 결과 | 100건 (`display=100`) |
| 최대 시작 위치 | 1,000번째 (`start=1000`) |
| 최대 수집 가능 건수 | 약 **1,100건** (start=1~1000, display=100) |

> **중요**: Client ID와 Client Secret은 **절대 공개하지 마세요.**
> GitHub 등에 올릴 때는 `.env` 파일에 저장하고 `.gitignore`에 추가하세요.

### API 문서 참조

| 검색 카테고리 | 문서 링크 |
|--------------|-----------|
| 블로그 | https://developers.naver.com/docs/serviceapi/search/blog/blog.md |
| 뉴스 | https://developers.naver.com/docs/serviceapi/search/news/news.md |
| 웹문서 | https://developers.naver.com/docs/serviceapi/search/web/web.md |

## 2. VSCode 환경 설정

### 1단계: 가상환경 생성 (터미널에서 실행)

```bash
cd my_project
python -m venv .venv

# Windows:
.venv\\Scripts\\activate
# macOS/Linux:
source .venv/bin/activate
```

### 2단계: 패키지 설치

```bash
pip install requests pandas openpyxl python-dotenv tqdm ipykernel
```

| 패키지 | 용도 |
|--------|------|
| `requests` | HTTP 요청 (API 호출) |
| `pandas` | 데이터프레임 (표 형태 정리) |
| `openpyxl` | Excel 파일 저장 |
| `python-dotenv` | `.env` 파일에서 환경 변수 로드 |
| `tqdm` | 진행률 표시 바 |
| `ipykernel` | Jupyter가 가상환경 Python을 커널로 인식 |

> 💡 **ipykernel**은 Jupyter가 가상환경의 Python을 커널로 인식하기 위해 반드시 필요합니다.

### 3단계: `.env` 파일 생성 (인증 정보 관리)

프로젝트 폴더에 `.env` 파일을 만들어 API 키를 안전하게 저장합니다.

```
# .env 파일 내용
NAVER_CLIENT_ID=발급받은_Client_ID
NAVER_CLIENT_SECRET=발급받은_Client_Secret
```

> **보안 팁**: `.gitignore` 파일에 `.env`를 추가하여 GitHub에 올라가지 않게 하세요.

### 4단계: VSCode 커널 선택

1. `.ipynb` 파일 열기
2. 우측 상단 **커널 선택** → `.venv` 선택

## 3. API 인증 설정

`.env` 파일에서 Client ID와 Client Secret을 불러옵니다.
`.env` 파일이 없으면 직접 입력받습니다.

In [ ]:
# ──────────────────────────────────────────────
# API 인증 정보 설정
#   .env 파일에서 로드하거나, 직접 입력
#   .env 파일 형식:
#     NAVER_CLIENT_ID=발급받은_ID
#     NAVER_CLIENT_SECRET=발급받은_SECRET
# ──────────────────────────────────────────────
import os
from dotenv import load_dotenv

# .env 파일에서 환경 변수 로드
#   .env 파일이 현재 디렉토리에 있으면 자동 로드
#   다른 경로에 있으면: load_dotenv("경로/.env")
load_dotenv()

# 환경 변수에서 인증 정보 가져오기
client_id     = os.getenv("NAVER_CLIENT_ID")
client_secret = os.getenv("NAVER_CLIENT_SECRET")

# 환경 변수가 없으면 직접 입력
if not client_id:
    client_id = input("네이버 API Client ID를 입력하세요: ")
if not client_secret:
    client_secret = input("네이버 API Client Secret을 입력하세요: ")

print("인증 정보가 설정되었습니다.")
print(f"Client ID: {client_id[:4]}{'*' * (len(client_id)-4)}")  # 앞 4자리만 표시

## 4. 첫 번째 API 호출: 웹문서 검색

가장 기본적인 API 호출을 단계별로 실습합니다.

### API 요청 구성 요소

| 구성 요소 | 설명 | 예시 |
|-----------|------|------|
| **URL** (엔드포인트) | API 서버 주소 | `https://openapi.naver.com/v1/search/blog.json` |
| **Headers** (헤더) | 인증 정보 | `X-Naver-Client-Id`, `X-Naver-Client-Secret` |
| **Params** (파라미터) | 검색 조건 | `query`, `display`, `start`, `sort` |

### 요청 파라미터 상세

| 파라미터 | 타입 | 필수 | 기본값 | 최댓값 | 설명 |
|----------|------|------|--------|--------|------|
| `query` | str | **필수** | - | - | 검색어 |
| `display` | int | 선택 | 10 | **100** | 한 번에 표시할 결과 수 |
| `start` | int | 선택 | 1 | **1000** | 검색 시작 위치 |
| `sort` | str | 선택 | `sim` | - | `sim`=정확도순, `date`=날짜순 |

> **참고**: `start`의 최댓값이 1000이고 `display`의 최댓값이 100이므로,
> 한 검색어당 최대 약 **1,100건**까지 수집 가능합니다.

In [ ]:
# ──────────────────────────────────────────────
# 첫 번째 API 호출: 웹문서 검색
#   요청 → 응답 → JSON 파싱 → 결과 확인
# ──────────────────────────────────────────────
import requests
import pandas as pd

# ---- 1) API 엔드포인트 (URL) ----
#   네이버 검색 API는 카테고리별로 다른 URL을 사용
url = "https://openapi.naver.com/v1/search/webkr.json"   # 웹문서 검색

# ---- 2) 요청 헤더 (인증 정보) ----
headers = {
    "X-Naver-Client-Id": client_id,
    "X-Naver-Client-Secret": client_secret,
}

# ---- 3) 요청 파라미터 (검색 조건) ----
params = {
    "query":   "경희대 맛집",   # 검색어
    "display": 10,              # 한 번에 표시할 결과 수 (최대 100)
    "start":   1,               # 검색 시작 위치 (1부터)
    "sort":    "sim",           # sim=정확도순, date=날짜순
}

# ---- 4) API 요청 ----
response = requests.get(url, headers=headers, params=params)

# ---- 5) 응답 확인 ----
print(f"상태 코드: {response.status_code}")    # 200이면 성공
print(f"응답 크기: {len(response.text):,} 글자")

## 5. 응답 데이터 분석

### JSON 응답 구조

```json
{
  "lastBuildDate": "Mon, 03 Mar 2025 10:00:00 +0900",
  "total": 12345,              ← 전체 검색 결과 건수
  "start": 1,                  ← 현재 시작 위치
  "display": 10,               ← 표시 건수
  "items": [                   ← 검색 결과 리스트
    {
      "title": "기사 제목",
      "link": "https://...",
      "description": "요약...",
      ...
    },
    ...
  ]
}
```

### 카테고리별 응답 필드

| 카테고리 | 주요 필드 |
|----------|-----------|
| 블로그 | title, link, description, **bloggername**, **bloggerlink**, **postdate** |
| 뉴스 | title, link, description, **originallink**, **pubDate** |
| 웹문서 | title, link, description |
| 책 | title, link, **author**, **publisher**, **isbn**, **image** |
| 쇼핑 | title, link, **lprice**, **hprice**, **mallName**, **image** |

In [ ]:
# ──────────────────────────────────────────────
# 응답 데이터 분석: JSON → DataFrame
# ──────────────────────────────────────────────

# JSON 파싱
result = response.json()

# ---- 응답 메타 정보 ----
print("=== 응답 메타 정보 ===")
print(f"  전체 검색 건수: {result['total']:,}건")
print(f"  시작 위치:      {result['start']}")
print(f"  표시 건수:      {result['display']}")
print(f"  빌드 시간:      {result['lastBuildDate']}")
print()

# ---- 검색 결과 확인 ----
items = result['items']
print(f"=== 검색 결과: {len(items)}건 ===")
print(f"  필드 목록: {list(items[0].keys())}")
print()

# 첫 번째 결과 상세 보기
print("=== 첫 번째 결과 ===")
for key, value in items[0].items():
    print(f"  {key:15s}: {str(value)[:60]}")
print()

# ---- DataFrame으로 변환 ----
df = pd.DataFrame(items)
print(f"DataFrame shape: {df.shape}")
df.head()

## 6. 검색 카테고리와 API 엔드포인트

네이버 검색 API는 **10가지 카테고리**를 지원합니다.
각 카테고리마다 **엔드포인트 URL**이 다릅니다.

| 카테고리 | 엔드포인트 URL | sort | start 최댓값 |
|----------|---------------|------|-------------|
| 블로그 | `/v1/search/blog.json` | sim, date | 1000 |
| 뉴스 | `/v1/search/news.json` | sim, date | 1000 |
| 책 | `/v1/search/book.json` | sim, date | 1000 |
| 백과사전 | `/v1/search/encyc.json` | ❌ 없음 | 100 |
| 카페글 | `/v1/search/cafearticle.json` | sim, date | 1000 |
| 지식iN | `/v1/search/kin.json` | sim, date | 1000 |
| 웹문서 | `/v1/search/webkr.json` | ❌ 없음 | 100 |
| 이미지 | `/v1/search/image` | sim, date | 1000 |
| 쇼핑 | `/v1/search/shop.json` | sim, date | 1000 |
| 전문자료 | `/v1/search/doc.json` | ❌ 없음 | 1000 |

> **주의**: 백과사전과 웹문서는 `start` 최댓값이 **100**으로,
> 다른 카테고리(1000)보다 수집 가능 범위가 좁습니다.
> `sort` 파라미터가 없는 카테고리는 정확도순 고정입니다.

In [ ]:
# ──────────────────────────────────────────────
# 검색 카테고리별 API 엔드포인트 URL 딕셔너리
# ──────────────────────────────────────────────

search_url_dct = {
    # ── 텍스트 검색 ──
    '블로그':   "https://openapi.naver.com/v1/search/blog.json",
    '뉴스':     "https://openapi.naver.com/v1/search/news.json",
    '책':       "https://openapi.naver.com/v1/search/book.json",
    '백과사전': "https://openapi.naver.com/v1/search/encyc.json",     # sort 없음, start 최댓값 100
    '카페글':   "https://openapi.naver.com/v1/search/cafearticle.json",
    '지식iN':   "https://openapi.naver.com/v1/search/kin.json",
    '웹문서':   "https://openapi.naver.com/v1/search/webkr.json",     # sort 없음, start 최댓값 100
    # ── 멀티미디어/쇼핑 ──
    '이미지':   "https://openapi.naver.com/v1/search/image",
    '쇼핑':     "https://openapi.naver.com/v1/search/shop.json",
    '전문자료': "https://openapi.naver.com/v1/search/doc.json",        # sort 없음
}

print("지원 카테고리:")
for name, url in search_url_dct.items():
    print(f"  {name:6s} → {url}")

## 7. 통합 검색 클래스 구현

여러 검색 카테고리를 **하나의 클래스**로 통합하고,
**페이지네이션(다중 페이지 수집)**을 자동으로 처리합니다.

### 클래스 설계

```python
naver_api = Naver_API(client_id="...", client_secret="...")

results_df = naver_api.search(
    where   = "블로그",        # 검색 카테고리
    query   = "경희대 맛집",   # 검색어
    display = 100,             # 페이지당 결과 수 (최대 100)
    start   = 1,               # 시작 위치
    pages   = 3,               # 수집할 페이지 수
    sort    = "sim"            # 정렬 (sim=정확도, date=날짜)
)
```

### 페이지네이션 원리

```
pages=3, display=100 일 때:
  페이지 1: start=1   → 1~100번째 결과
  페이지 2: start=101 → 101~200번째 결과
  페이지 3: start=201 → 201~300번째 결과
  → 총 300건 수집
```

| pages | display=100 기준 수집량 | start 값 |
|-------|----------------------|-----------|
| 1 | 100건 | 1 |
| 3 | 300건 | 1, 101, 201 |
| 5 | 500건 | 1, 101, ..., 401 |
| 10 | 1,000건 (최대) | 1, 101, ..., 901 |

> **최대 수집량**: `pages=10`, `display=100`이면 start가 901까지 → **약 1,000건**

In [ ]:
# ──────────────────────────────────────────────
# 네이버 API 통합 검색 클래스
#   다양한 카테고리 지원 + 자동 페이지네이션
# ──────────────────────────────────────────────
import requests
import pandas as pd
import time
from tqdm import tqdm


class Naver_API:
    """
    네이버 검색 API를 사용해 다양한 카테고리의 검색 결과를 수집하는 클래스.

    사용 전 준비사항:
        1. 네이버 개발자 센터(https://developers.naver.com)에서 애플리케이션 등록
        2. 검색 API 서비스 신청
        3. 발급받은 Client ID와 Client Secret 사용

    사용 예시:
        api = Naver_API(client_id="...", client_secret="...")
        df  = api.search(where="블로그", query="검색어", display=100, pages=3)
    """

    def __init__(self, client_id: str, client_secret: str):
        """
        Parameters:
            client_id     : 네이버 API Client ID
            client_secret : 네이버 API Client Secret
        """
        self.client_id = client_id
        self.client_secret = client_secret
        self.headers = {
            "X-Naver-Client-Id": self.client_id,
            "X-Naver-Client-Secret": self.client_secret,
        }
        print("Naver_API 인스턴스 생성 완료")

    def search(self, **kwargs) -> pd.DataFrame:
        """
        네이버 검색 API를 호출하여 결과를 DataFrame으로 반환합니다.

        Parameters (kwargs):
            where   : str  - 검색 카테고리 (search_url_dct 키)
            query   : str  - 검색어
            display : int  - 페이지당 결과 수 (기본 10, 최대 100)
            start   : int  - 시작 위치 (기본 1)
            pages   : int  - 수집할 페이지 수 (기본 1, 최대 10)
            sort    : str  - 정렬 (sim=정확도, date=날짜)

        Returns:
            pd.DataFrame : 검색 결과
        """
        # ---- 카테고리 → URL 매핑 ----
        where = kwargs.get('where', '웹문서')
        if where not in search_url_dct:
            print(f"오류: '{where}'은(는) 지원하지 않는 카테고리입니다.")
            print(f"지원 카테고리: {list(search_url_dct.keys())}")
            return pd.DataFrame()

        url = search_url_dct[where]
        pages = kwargs.get('pages', 1)
        display = kwargs.get('display', 100)

        # ---- 첫 요청으로 전체 건수 확인 ----
        #   where, pages 등은 네이버 API 파라미터가 아니므로
        #   API에 보낼 파라미터만 별도로 구성
        first_params = {
            "query":   kwargs.get('query', ''),
            "display": 1,                           # 건수 확인만이므로 1건만 요청
            "start":   1,
            "sort":    kwargs.get('sort', 'sim'),
        }
        response = requests.get(url, headers=self.headers, params=first_params)
        time.sleep(0.5)

        if response.status_code != 200:
            print(f"API 요청 실패: 상태 코드 {response.status_code}")
            print(f"응답: {response.text[:200]}")
            return pd.DataFrame()

        result = response.json()

        if 'items' not in result or not result['items']:
            print("검색 결과가 없습니다.")
            return pd.DataFrame()

        columns = list(result['items'][0].keys())
        print(f"전체 검색 건수: {result['total']:,}건")

        # ---- 페이지별 수집 ----
        all_results = []

        for page in tqdm(range(pages), desc=f"{where} 수집"):
            start = page * display + 1     # 1, 101, 201, ...

            # start 최댓값 체크 (1000 초과 불가)
            if start > 1000:
                print(f"\nstart={start} → 최댓값(1000) 초과. 수집 종료.")
                break

            params = {
                "query":   kwargs.get('query', ''),
                "display": display,
                "start":   start,
                "sort":    kwargs.get('sort', 'sim'),
            }

            response = requests.get(url, headers=self.headers, params=params)
            time.sleep(1)   # 서버 부담 방지

            if response.status_code != 200:
                print(f"\n페이지 {page+1} 요청 실패: {response.status_code}")
                continue

            try:
                result = response.json()
                items = result.get('items', [])
                if not items:
                    print(f"\n페이지 {page+1}: 결과 없음. 수집 종료.")
                    break

                for item in items:
                    all_results.append(list(item.values()))

            except Exception as e:
                print(f"\n페이지 {page+1} 처리 오류: {e}")
                continue

        # ---- DataFrame 생성 ----
        if all_results:
            df = pd.DataFrame(all_results, columns=columns)
            print(f"\n총 {len(df)}건 수집 완료.")
            return df
        else:
            print("수집된 데이터가 없습니다.")
            return pd.DataFrame(columns=columns)


print("Naver_API 클래스 정의 완료")

## 8. 실행 및 결과 저장

클래스를 사용하여 검색을 실행하고, 결과를 Excel로 저장합니다.

### 사용 예시

```python
# 인스턴스 생성
api = Naver_API(client_id=client_id, client_secret=client_secret)

# 검색 실행
df = api.search(
    where   = "블로그",
    query   = "경희대 맛집",
    display = 100,
    pages   = 3,
    sort    = "sim"
)

# Excel 저장
df.to_excel("결과.xlsx", index=False)
```

In [ ]:
# ──────────────────────────────────────────────
# 검색 실행: 블로그 검색
# ──────────────────────────────────────────────

# ---- 인스턴스 생성 ----
naver_api = Naver_API(
    client_id=client_id,
    client_secret=client_secret
)

# ---- 검색 조건 설정 ----
kwargs = {
    'where':   '블로그',         # 검색 카테고리 (10가지 중 선택)
    'query':   '경희대 맛집',    # 검색어
    'display': 100,              # 페이지당 결과 수 (최대 100, 수정 불필요)
    'start':   1,                # 시작 위치 (기본 1, 수정 불필요)
    'pages':   3,                # 수집 페이지 수 (최대 10)
    'sort':    'sim',            # sim=정확도순, date=날짜순
}

# ---- 검색 실행 ----
results_df = naver_api.search(**kwargs)

# ---- 결과 미리보기 ----
results_df.head()

In [ ]:
# ──────────────────────────────────────────────
# 결과 저장 (Excel)
# ──────────────────────────────────────────────

filename = f"네이버{kwargs['where']}_{kwargs['query'].replace(' ', '_')}.xlsx"
results_df.to_excel(filename, index=False)
print(f"저장 완료: {filename}")
print(f"총 {len(results_df)}건")

In [ ]:
# ──────────────────────────────────────────────
# 다른 카테고리 검색 예시: 뉴스
# ──────────────────────────────────────────────

news_df = naver_api.search(
    where   = '뉴스',
    query   = '인공지능',
    display = 100,
    pages   = 2,
    sort    = 'date'         # 날짜순 정렬
)

news_df.head()

---

## 네이버 검색 API 핵심 요약 (Quick Reference)

### API 인증

```python
headers = {
    "X-Naver-Client-Id": "발급받은_ID",
    "X-Naver-Client-Secret": "발급받은_SECRET",
}
response = requests.get(url, headers=headers, params=params)
```

### 요청 파라미터

| 파라미터 | 타입 | 필수 | 기본값 | 최댓값 | 설명 |
|----------|------|------|--------|--------|------|
| `query` | str | **필수** | - | - | 검색어 |
| `display` | int | 선택 | 10 | 100 | 표시할 결과 수 |
| `start` | int | 선택 | 1 | 1000 | 검색 시작 위치 |
| `sort` | str | 선택 | sim | - | sim=정확도, date=날짜 |

### 검색 카테고리별 엔드포인트

| 카테고리 | URL |
|----------|-----|
| 블로그 | `openapi.naver.com/v1/search/blog.json` |
| 뉴스 | `openapi.naver.com/v1/search/news.json` |
| 책 | `openapi.naver.com/v1/search/book.json` |
| 카페글 | `openapi.naver.com/v1/search/cafearticle.json` |
| 지식iN | `openapi.naver.com/v1/search/kin.json` |
| 웹문서 | `openapi.naver.com/v1/search/webkr.json` |
| 이미지 | `openapi.naver.com/v1/search/image` |
| 쇼핑 | `openapi.naver.com/v1/search/shop.json` |

### 응답 JSON 구조

```python
result = response.json()
result['total']     # 전체 검색 건수
result['start']     # 현재 시작 위치
result['display']   # 표시 건수
result['items']     # 검색 결과 리스트 (각 항목은 딕셔너리)
```

### 카테고리별 주요 응답 필드

| 카테고리 | 고유 필드 |
|----------|-----------|
| 블로그 | `bloggername`, `bloggerlink`, `postdate` |
| 뉴스 | `originallink`, `pubDate` |
| 책 | `author`, `publisher`, `isbn`, `image`, `price` |
| 쇼핑 | `lprice`, `hprice`, `mallName`, `image` |

### 페이지네이션 (다중 페이지 수집)

```python
# display=100, pages=10 → 최대 약 1,000건
for page in range(pages):
    start = page * display + 1    # 1, 101, 201, ..., 901
    params["start"] = start
    response = requests.get(url, headers=headers, params=params)
    time.sleep(1)                  # 서버 부담 방지
```

### 에러 코드

| 코드 | 의미 |
|------|------|
| 200 | 성공 |
| 400 | 잘못된 요청 (파라미터 오류) |
| 401 | 인증 실패 (Client ID/Secret 오류) |
| 403 | 권한 없음 |
| 429 | 일일 호출 한도 초과 (25,000건) |
| 500 | 서버 오류 |

### .env 파일 관리

```bash
# .env 파일 (프로젝트 루트)
NAVER_CLIENT_ID=발급받은_ID
NAVER_CLIENT_SECRET=발급받은_SECRET
```

```python
# Python에서 로드
from dotenv import load_dotenv
import os
load_dotenv()
client_id = os.getenv("NAVER_CLIENT_ID")
```